# A07. Z-Score와 표준화

**학습 목표**
- 원자료를 Z-score로 변환하는 이유와 의미를 이해한다 (통계학적, 머신러닝 관점)
- 원자료 → 평균중심화 → 표준화의 3단계 변환을 실습한다
- `scipy.stats.zscore()`를 활용한 표준화를 익힌다
- 타이타닉 데이터를 활용하여 표준화 전/후를 비교한다

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import binom, bernoulli, norm, uniform, expon, pareto
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
sns.set_palette('Set2')

np.random.seed(42)
print('라이브러리 로드 완료')

---
## 1. 샘플 데이터 생성

한국 성인 남성 키 데이터를 정규분포로 시뮬레이션한다:

$$X \sim N(\mu = 170, \sigma = 10)$$

- 평균($\mu$): 170cm
- 표준편차($\sigma$): 10cm
- 표본 크기: 100명

In [ ]:
# === 샘플 데이터 생성 ===
height_data = np.random.normal(170, 10, 100)  # 키 데이터 (cm)

print('=== 원자료 (키 데이터, 단위: cm) ===')
print(f'표본 크기: {len(height_data)}')
print(f'평균: {np.mean(height_data):.2f} cm')
print(f'표준편차: {np.std(height_data):.2f} cm')
print(f'최솟값: {np.min(height_data):.2f} cm')
print(f'최댓값: {np.max(height_data):.2f} cm')
print(f'\n처음 10개: {np.round(height_data[:10], 2)}')

---
## 2. 3단계 변환: 원자료 → 평균중심화 → 표준화

### 변환 과정

| 단계 | 이름 | 수식 | 결과 | 의미 |
|------|------|------|------|------|
| 1 | **원자료** | $X$ | $\mu \approx 170$, $\sigma \approx 10$ | 원래 데이터 |
| 2 | **평균중심화** | $X - \mu$ | $\mu = 0$, $\sigma \approx 10$ | 평균을 0으로 이동 |
| 3 | **표준화** | $Z = \frac{X - \mu}{\sigma}$ | $\mu = 0$, $\sigma = 1$ | 단위까지 통일 |

In [ ]:
# === 3단계 변환 수행 ===
mu = np.mean(height_data)
sigma = np.std(height_data)

# 단계 1: 원자료 (Original Data)
step1_original = height_data.copy()

# 단계 2: 평균중심화 (Mean Centering): X - μ
step2_centered = height_data - mu

# 단계 3: 표준화 (Standardization): Z = (X - μ) / σ
step3_standardized = (height_data - mu) / sigma

# 각 단계 통계량 비교
comparison = pd.DataFrame({
    '단계': ['① 원자료 (X)', '② 평균중심화 (X-μ)', '③ 표준화 (Z-score)'],
    '평균': [np.mean(step1_original), np.mean(step2_centered), np.mean(step3_standardized)],
    '표준편차': [np.std(step1_original), np.std(step2_centered), np.std(step3_standardized)],
    '최솟값': [np.min(step1_original), np.min(step2_centered), np.min(step3_standardized)],
    '최댓값': [np.max(step1_original), np.max(step2_centered), np.max(step3_standardized)]
})
print('=== 3단계 변환 통계량 비교 ===')
print(comparison.to_string(index=False))

In [ ]:
# === 각 단계별 분포 시각화 (3개 subplot 나란히) ===
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

datasets = [
    (step1_original, '① 원자료 (X)', 'cm', sns.color_palette('Set2')[0]),
    (step2_centered, '② 평균중심화 (X - μ)', 'cm (편차)', sns.color_palette('Set2')[1]),
    (step3_standardized, '③ 표준화 (Z-score)', 'σ 단위', sns.color_palette('Set2')[2])
]

for idx, (data, title, xlabel, color) in enumerate(datasets):
    ax = axes[idx]
    
    # 히스토그램
    ax.hist(data, bins=15, density=True, color=color, 
            edgecolor='black', alpha=0.7)
    
    # 정규분포 곡선 오버레이
    x_range = np.linspace(data.min() - 1, data.max() + 1, 200)
    ax.plot(x_range, norm.pdf(x_range, np.mean(data), np.std(data)),
            'r-', linewidth=2.5, label='정규분포 곡선')
    
    # 평균선
    ax.axvline(np.mean(data), color='darkred', linestyle='--', linewidth=2,
               label=f'평균 = {np.mean(data):.2f}')
    
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel('밀도', fontsize=12)
    ax.set_title(f'{title}\n$\mu$={np.mean(data):.2f}, $\sigma$={np.std(data):.2f}',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('원자료 → 평균중심화 → 표준화 비교', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 해석

- **① 원자료**: 평균이 약 170, 표준편차가 약 10인 분포. 단위는 cm.
- **② 평균중심화**: 평균이 0으로 이동했지만, **퍼진 정도(σ)는 변하지 않았다**. 아직 cm 단위.
- **③ 표준화**: 평균 0, 표준편차 1로 완전히 변환. **단위가 사라지고** σ 단위로 표현된다.

> **핵심 차이**: 평균중심화는 **위치(location)만** 변경하고, 표준화는 **위치와 스케일(scale) 모두** 변경한다.

---
## 3. Z-Score의 통계학적 의미

### Z-Score 공식

$$Z = \frac{X - \mu}{\sigma}$$

### 해석 방법

| Z-Score | 의미 | 백분위 (대략) |
|---------|------|---------------|
| $Z = 0$ | 정확히 평균 위치 | 50% |
| $Z = +1$ | 평균보다 1 표준편차 위 | 84.1% |
| $Z = -1$ | 평균보다 1 표준편차 아래 | 15.9% |
| $Z = +2$ | 평균보다 2 표준편차 위 | 97.7% |
| $Z = -2$ | 평균보다 2 표준편차 아래 | 2.3% |

### σ-구간 해석

- $-1 \leq Z \leq +1$: 전체의 약 **68%**가 포함
- $-2 \leq Z \leq +2$: 전체의 약 **95%**가 포함
- $-3 \leq Z \leq +3$: 전체의 약 **99.7%**가 포함

In [ ]:
# === Z-Score의 σ-구간 시각화 ===
fig, ax = plt.subplots(figsize=(14, 7))

x = np.linspace(-4, 4, 1000)
y = norm.pdf(x, 0, 1)

# 전체 곡선
ax.plot(x, y, 'k-', linewidth=2)

# 3σ 영역 (99.7%) - 가장 넓은 영역부터 채우기
x_3sigma = np.linspace(-3, 3, 500)
ax.fill_between(x_3sigma, norm.pdf(x_3sigma, 0, 1), alpha=0.15,
                color='green', label=f'±3σ: {norm.cdf(3)-norm.cdf(-3):.4f} (99.7%)')

# 2σ 영역 (95.4%)
x_2sigma = np.linspace(-2, 2, 500)
ax.fill_between(x_2sigma, norm.pdf(x_2sigma, 0, 1), alpha=0.2,
                color='blue', label=f'±2σ: {norm.cdf(2)-norm.cdf(-2):.4f} (95.4%)')

# 1σ 영역 (68.3%)
x_1sigma = np.linspace(-1, 1, 500)
ax.fill_between(x_1sigma, norm.pdf(x_1sigma, 0, 1), alpha=0.3,
                color='red', label=f'±1σ: {norm.cdf(1)-norm.cdf(-1):.4f} (68.3%)')

# σ 경계선
for z_val in [-3, -2, -1, 0, 1, 2, 3]:
    ax.axvline(z_val, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)
    ax.text(z_val, -0.02, f'{z_val}σ', ha='center', fontsize=10, fontweight='bold')

# 원자료 스케일 (키 데이터) 보조 축
ax2 = ax.twiny()
ax2.set_xlim(170 - 4*10, 170 + 4*10)
ax2.set_xlabel('키 (cm) — 원자료 스케일', fontsize=11, color='brown')
ax2.tick_params(axis='x', labelcolor='brown')

ax.set_xlabel('Z-Score (표준편차 단위)', fontsize=13)
ax.set_ylabel('확률 밀도', fontsize=13)
ax.set_title('표준정규분포의 σ-구간 (68-95-99.7 법칙)', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()

### 해석

- **Z=1 (180cm)**: 평균+1 표준편차 위치. 전체의 약 84.1%보다 크다 → 상위 약 16%.
- **Z=-1 (160cm)**: 평균-1 표준편차 위치. 전체의 약 15.9%보다 크다 → 하위 약 16%.
- **Z=2 (190cm)**: 매우 키가 큰 편. 상위 약 2.3%.
- 하단 보조 축에서 Z-score와 원래 키(cm) 스케일의 관계를 확인할 수 있다.

> **단위 없는 비교**: Z-score로 변환하면 cm, kg, 점수 등 **서로 다른 단위의 데이터를 동일한 기준으로 비교**할 수 있다.

In [ ]:
# === 구체적인 예시: 키 180cm인 사람의 Z-score ===
height_example = 180
z_example = (height_example - mu) / sigma

print(f'=== Z-Score 계산 예시 ===')
print(f'데이터: 키 = {height_example} cm')
print(f'표본 평균(μ) = {mu:.2f} cm')
print(f'표본 표준편차(σ) = {sigma:.2f} cm')
print(f'\nZ = (X - μ) / σ = ({height_example} - {mu:.2f}) / {sigma:.2f} = {z_example:.4f}')
print(f'\n→ 이 사람의 키는 평균보다 {z_example:.2f} 표준편차만큼 크다')
print(f'→ 백분위: {norm.cdf(z_example)*100:.1f}% (상위 {(1-norm.cdf(z_example))*100:.1f}%)')

---
## 4. 머신러닝 관점에서의 표준화

### 왜 ML에서 표준화가 중요한가?

#### 4.1 특성 스케일링 (Feature Scaling)
- 서로 다른 스케일의 특성이 있을 때, 스케일이 큰 특성이 모델에 과도한 영향을 미침
- 예: 키(150~190cm) vs 몸무게(40~100kg) → 스케일 차이가 크다

#### 4.2 경사하강법(Gradient Descent) 수렴 속도
- 특성 스케일이 불균일하면 **타원형 등고선** → 수렴이 느리고 지그재그
- 표준화하면 **원형 등고선** → 최적해에 빠르게 도달

#### 4.3 거리 기반 알고리즘
- KNN, K-Means 등은 유클리드 거리를 사용 → 스케일에 민감

In [ ]:
# === 다른 단위의 변수 비교: 키(cm) vs 몸무게(kg) ===
np.random.seed(42)
n_people = 100

# 키와 몸무게 데이터 (양의 상관관계)
height = np.random.normal(170, 10, n_people)  # cm
weight = 0.6 * height + np.random.normal(-30, 5, n_people)  # kg (키와 상관)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# (1) 원자료: 스케일이 다름
axes[0].scatter(height, weight, c=sns.color_palette('Set2')[0], 
                edgecolor='black', alpha=0.6, s=50)
axes[0].set_xlabel('키 (cm)', fontsize=12)
axes[0].set_ylabel('몸무게 (kg)', fontsize=12)
axes[0].set_title('① 원자료\n(스케일이 다름)', fontsize=13, fontweight='bold')
axes[0].set_aspect('equal')

# (2) 원자료: 같은 축 범위로
axes[1].scatter(height, weight, c=sns.color_palette('Set2')[1], 
                edgecolor='black', alpha=0.6, s=50)
axes[1].set_xlabel('키 (cm)', fontsize=12)
axes[1].set_ylabel('몸무게 (kg)', fontsize=12)
axes[1].set_title('② 원자료\n(자동 스케일)', fontsize=13, fontweight='bold')

# (3) 표준화 후: 동일 스케일
height_z = stats.zscore(height)
weight_z = stats.zscore(weight)
axes[2].scatter(height_z, weight_z, c=sns.color_palette('Set2')[2], 
                edgecolor='black', alpha=0.6, s=50)
axes[2].set_xlabel('키 (Z-score)', fontsize=12)
axes[2].set_ylabel('몸무게 (Z-score)', fontsize=12)
axes[2].set_title('③ 표준화 후\n(동일 스케일)', fontsize=13, fontweight='bold')
axes[2].set_aspect('equal')
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.3)
axes[2].axvline(0, color='gray', linestyle='--', alpha=0.3)

plt.suptitle('표준화 전후 비교: 키(cm) vs 몸무게(kg)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('=== 표준화 전후 통계량 ===')
comp_df = pd.DataFrame({
    '변수': ['키 (원자료)', '키 (표준화)', '몸무게 (원자료)', '몸무게 (표준화)'],
    '평균': [np.mean(height), np.mean(height_z), np.mean(weight), np.mean(weight_z)],
    '표준편차': [np.std(height), np.std(height_z), np.std(weight), np.std(weight_z)],
    '최소': [np.min(height), np.min(height_z), np.min(weight), np.min(weight_z)],
    '최대': [np.max(height), np.max(height_z), np.max(weight), np.max(weight_z)]
})
print(comp_df.to_string(index=False))

### 해석

- **① 등비율 축**: 키(150~190)와 몸무게(60~90)의 절대 스케일 차이로 패턴 파악이 어렵다.
- **② 자동 스케일**: matplotlib이 자동 조정하여 상관관계가 보이지만, 두 변수의 "기여도"가 불균등하다.
- **③ 표준화 후**: 두 변수 모두 평균 0, 표준편차 1로 맞춰져 **공정한 비교**가 가능하다. 원점(0,0)이 평균이 된다.

In [ ]:
# === 경사하강법 수렴 속도 시각화 (개념적) ===
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# (1) 표준화 전: 타원형 등고선 → 지그재그 경로
from matplotlib.patches import Ellipse

# 타원형 등고선
for scale in [1, 2, 3, 4]:
    ellipse = Ellipse((170, 70), width=scale*20, height=scale*5,
                       fill=False, edgecolor='blue', linewidth=1.5, alpha=0.5)
    axes[0].add_patch(ellipse)

# 지그재그 경사하강 경로
path_x = [190, 175, 182, 172, 175, 171, 172, 170.5, 170.8, 170]
path_y = [85, 68, 75, 69, 72, 70, 71, 70.2, 70.3, 70]
axes[0].plot(path_x, path_y, 'r.-', markersize=8, linewidth=1.5, label='경사하강 경로')
axes[0].plot(170, 70, 'r*', markersize=15, zorder=5)
axes[0].set_xlabel('키 (cm)', fontsize=12)
axes[0].set_ylabel('몸무게 (kg)', fontsize=12)
axes[0].set_title('표준화 전: 지그재그 수렴\n(타원형 등고선)', fontsize=13, fontweight='bold')
axes[0].set_xlim(150, 195)
axes[0].set_ylim(55, 90)
axes[0].legend(fontsize=10)

# (2) 표준화 후: 원형 등고선 → 직선 경로
for scale in [0.5, 1, 1.5, 2]:
    circle = plt.Circle((0, 0), scale, fill=False, edgecolor='blue',
                         linewidth=1.5, alpha=0.5)
    axes[1].add_patch(circle)

# 직선적 경사하강 경로
path_x2 = [2, 1.4, 0.9, 0.5, 0.25, 0.1, 0.03, 0]
path_y2 = [1.8, 1.2, 0.7, 0.35, 0.15, 0.05, 0.01, 0]
axes[1].plot(path_x2, path_y2, 'r.-', markersize=8, linewidth=1.5, label='경사하강 경로')
axes[1].plot(0, 0, 'r*', markersize=15, zorder=5)
axes[1].set_xlabel('키 (Z-score)', fontsize=12)
axes[1].set_ylabel('몸무게 (Z-score)', fontsize=12)
axes[1].set_title('표준화 후: 빠른 수렴\n(원형 등고선)', fontsize=13, fontweight='bold')
axes[1].set_xlim(-3, 3)
axes[1].set_ylim(-3, 3)
axes[1].set_aspect('equal')
axes[1].legend(fontsize=10)

plt.suptitle('표준화가 경사하강법 수렴에 미치는 영향', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 해석

- **표준화 전**: 특성 스케일이 다르면 손실 함수의 등고선이 **타원형**이 되어, 경사하강법이 지그재그로 비효율적으로 수렴한다.
- **표준화 후**: 등고선이 **원형**에 가까워져, 경사하강법이 최적해를 향해 **직선적으로 빠르게** 수렴한다.
- 이것이 실무에서 **Standard Scaler** 또는 **Z-score 정규화**를 전처리 단계에서 적용하는 이유이다.

---
## 5. scipy.stats.zscore() 활용

In [ ]:
# === scipy.stats.zscore() 사용법 ===
from scipy.stats import zscore

# 수동 계산
z_manual = (height_data - np.mean(height_data)) / np.std(height_data)

# scipy 자동 계산 (ddof=0이 기본, 모분산 사용)
z_scipy = zscore(height_data)

# 결과 비교
print('=== scipy.stats.zscore() vs 수동 계산 ===')
print(f'수동 계산 (처음 5개): {np.round(z_manual[:5], 6)}')
print(f'scipy 계산 (처음 5개): {np.round(z_scipy[:5], 6)}')
print(f'\n두 결과의 차이 최대값: {np.max(np.abs(z_manual - z_scipy)):.15f}')
print(f'→ 사실상 동일한 결과 (부동소수점 오차 수준)')

# 검증: 평균=0, 표준편차=1
print(f'\nZ-score 평균: {np.mean(z_scipy):.10f} (≈ 0)')
print(f'Z-score 표준편차: {np.std(z_scipy):.10f} (≈ 1)')

---
## 6. 실습: 타이타닉 Age vs Fare 표준화 전/후 비교

In [ ]:
# === 타이타닉 데이터 로드 (seaborn 내장) ===
titanic = sns.load_dataset('titanic')

# age와 fare 열 추출 (결측치 제거)
titanic_clean = titanic[['age', 'fare']].dropna()

print('=== 타이타닉 데이터 기본 정보 ===')
print(f'데이터 수: {len(titanic_clean)}')
print(f'\nAge (나이):  평균={titanic_clean["age"].mean():.2f}, '
      f'표준편차={titanic_clean["age"].std():.2f}, '
      f'범위=[{titanic_clean["age"].min():.1f}, {titanic_clean["age"].max():.1f}]')
print(f'Fare (요금): 평균={titanic_clean["fare"].mean():.2f}, '
      f'표준편차={titanic_clean["fare"].std():.2f}, '
      f'범위=[{titanic_clean["fare"].min():.1f}, {titanic_clean["fare"].max():.1f}]')
print(f'\n→ Age는 0~80 범위, Fare는 0~512 범위 → 스케일이 매우 다르다!')

In [ ]:
# === 타이타닉 Age vs Fare: 표준화 전/후 비교 ===
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# (1) 표준화 전
scatter1 = axes[0].scatter(titanic_clean['age'], titanic_clean['fare'],
                            c=sns.color_palette('Set2')[0], alpha=0.5,
                            edgecolor='gray', s=30)
axes[0].set_xlabel('Age (나이, 세)', fontsize=13)
axes[0].set_ylabel('Fare (운임, $)', fontsize=13)
axes[0].set_title('표준화 전: Age vs Fare\n(스케일 불일치)', fontsize=14, fontweight='bold')

# 평균 표시
axes[0].axvline(titanic_clean['age'].mean(), color='red', linestyle='--', alpha=0.5,
                label=f'Age 평균 = {titanic_clean["age"].mean():.1f}')
axes[0].axhline(titanic_clean['fare'].mean(), color='blue', linestyle='--', alpha=0.5,
                label=f'Fare 평균 = {titanic_clean["fare"].mean():.1f}')
axes[0].legend(fontsize=10)

# (2) 표준화 후
age_z = zscore(titanic_clean['age'])
fare_z = zscore(titanic_clean['fare'])

scatter2 = axes[1].scatter(age_z, fare_z,
                            c=sns.color_palette('Set2')[2], alpha=0.5,
                            edgecolor='gray', s=30)
axes[1].set_xlabel('Age (Z-score)', fontsize=13)
axes[1].set_ylabel('Fare (Z-score)', fontsize=13)
axes[1].set_title('표준화 후: Age vs Fare\n(동일 스케일)', fontsize=14, fontweight='bold')

# 원점 표시
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5, label='평균 (Z=0)')
axes[1].axhline(0, color='blue', linestyle='--', alpha=0.5)

# σ 범위 표시
for z_boundary in [-2, -1, 1, 2]:
    axes[1].axvline(z_boundary, color='gray', linestyle=':', alpha=0.2)
    axes[1].axhline(z_boundary, color='gray', linestyle=':', alpha=0.2)

axes[1].legend(fontsize=10)

plt.suptitle('타이타닉 데이터: 표준화 전후 비교', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# 표준화 전후 통계량
print('\n=== 표준화 전후 통계량 비교 ===')
stats_df = pd.DataFrame({
    '변수': ['Age (원자료)', 'Age (Z-score)', 'Fare (원자료)', 'Fare (Z-score)'],
    '평균': [titanic_clean['age'].mean(), np.mean(age_z),
            titanic_clean['fare'].mean(), np.mean(fare_z)],
    '표준편차': [titanic_clean['age'].std(), np.std(age_z),
              titanic_clean['fare'].std(), np.std(fare_z)],
    '최소': [titanic_clean['age'].min(), np.min(age_z),
            titanic_clean['fare'].min(), np.min(fare_z)],
    '최대': [titanic_clean['age'].max(), np.max(age_z),
            titanic_clean['fare'].max(), np.max(fare_z)]
})
print(stats_df.to_string(index=False))

### 해석

- **표준화 전**: Age(0~80)와 Fare(0~512)의 스케일 차이가 매우 크다. KNN이나 SVM 같은 거리 기반 모델에서는 Fare가 Age보다 압도적으로 큰 영향을 미친다.
- **표준화 후**: 두 변수 모두 평균 0, 표준편차 1로 동일 스케일이 되었다. Z=0이 평균, Z=±1이 1 표준편차 범위.
- Fare의 Z-score가 8 이상인 극단값(이상치)이 보인다. 이는 1등석 고가 티켓 승객이다.

> **실무 팁**: 이상치가 많은 데이터에서는 Z-score 대신 **Robust Scaling** (중앙값/IQR 기반)을 고려해야 할 수 있다.

---
## 7. 정리

### 표준화(Z-Score) 요약

| 관점 | 내용 |
|------|------|
| **공식** | $Z = \frac{X - \mu}{\sigma}$ |
| **결과** | 평균 = 0, 표준편차 = 1 |
| **통계학적 의미** | 단위 제거 → 서로 다른 변수 비교 가능, σ-구간으로 위치 해석 |
| **ML 의미** | 특성 스케일링, 경사하강법 수렴 가속, 거리 기반 알고리즘 공정성 |
| **Python** | `scipy.stats.zscore()` 또는 `sklearn.preprocessing.StandardScaler` |

### 3단계 변환 정리

1. **원자료** ($X$): 원래 단위와 스케일 유지
2. **평균중심화** ($X - \mu$): 평균을 0으로 이동. 퍼진 정도(σ)는 유지.
3. **표준화** ($Z = \frac{X-\mu}{\sigma}$): 평균 0, 표준편차 1로 완전 변환. **단위가 사라진다.**